# Querying Your Own Data + Building Vector Store

**Hands-on + theory (token-aware chunking, embeddings, FAISS vector store, retrieval, grounding-ready outputs)**

## Overview
In this lab, you will implement the core retrieval workflow used in most Retrieval-Augmented Generation (RAG) systems: ingest your own documents, split them into **token-aware chunks**, generate **embeddings** using Azure OpenAI, store them in a **vector index**, and run **semantic queries** to retrieve the most relevant passages. You’ll also add practical improvements like metadata filtering, chunk quality checks, and MMR diversification. Finally, you’ll package results into a **grounding-ready bundle** (citations) and (optionally) generate grounded answers.

## Learning goals
By the end, you will be able to:
- Explain “querying your own data” and why retrieval is needed.
- Build token-aware chunking with metadata.
- Generate embeddings with Azure OpenAI and index them in FAISS.
- Retrieve top-k semantic matches and interpret scores.
- Improve retrieval quality with filtering, dedup, minimum chunk length, and MMR.
- Produce grounding-ready results suitable for answer generation.

## Expected prerequisites
- Comfort with Python and Jupyter
- Access to an Azure OpenAI resource + deployments

## Estimated time
~120 minutes

---

### Security reminder
Do **not** hardcode API keys in notebooks. Use environment variables or a secret manager.


# 0) Setup

We will use:
- `openai` (AzureOpenAI client)
- `tiktoken` (token counting and chunking)
- `numpy`, `pandas` (vector math + inspection)
- `faiss-cpu` (local vector store/index)

> If you’re running in a managed notebook environment, you may need to restart the kernel after installing packages.


In [1]:
%pip install -q --upgrade openai tiktoken numpy pandas faiss-cpu scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


## 0.1 Configure Azure OpenAI securely

Set these environment variables before running the lab:
- `AZURE_OPENAI_ENDPOINT`  → `https://<your-resource>.openai.azure.com/`
- `AZURE_OPENAI_API_KEY`   → your Azure OpenAI key
- `AZURE_OPENAI_API_VERSION` (optional) → defaults to `2024-12-01-preview`
- `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` → e.g., `text-embedding-ada-002`
- `AZURE_OPENAI_CHAT_DEPLOYMENT` (optional) → e.g., `gpt-4o-mini`

This notebook contains **no secrets**. Keep keys out of code.


In [2]:
import os
from openai import AzureOpenAI


AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', 'https://<TBD>.cognitiveservices.azure.com/').strip()
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '').strip()
API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')

EMBEDDINGS_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT', 'text-embedding-ada-002')
CHAT_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT', 'gpt-4o')

if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_API_KEY:
    raise ValueError('Missing AZURE_OPENAI_ENDPOINT or AZURE_OPENAI_API_KEY. Set them as environment variables.')

client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)

print('AzureOpenAI client ready')
print('API version:', API_VERSION)
print('Embeddings deployment:', EMBEDDINGS_DEPLOYMENT)
print('Chat deployment (optional):', CHAT_DEPLOYMENT)

AzureOpenAI client ready
API version: 2024-12-01-preview
Embeddings deployment: text-embedding-ada-002
Chat deployment (optional): gpt-4o


## 0.2 Connectivity check (embeddings)

We’ll embed a tiny string and confirm we get a vector back.


In [3]:
import numpy as np

resp = client.embeddings.create(
    model=EMBEDDINGS_DEPLOYMENT,
    input=['quick connectivity test'],
)
vec = np.array(resp.data[0].embedding, dtype=np.float32)
print('Embedding received')
print('Vector length:', vec.shape[0])
print('First 8 dims:', vec[:8])

Embedding received
Vector length: 1536
First 8 dims: [-0.02028767  0.00808607  0.01557828 -0.01137297 -0.01767748  0.00912876
 -0.00619057 -0.00572447]


# 1) Querying your own data (theory)

## What it means
“Querying your own data” means answering user questions using **your private or domain-specific documents** (policies, manuals, wikis, PDFs, tickets), rather than relying only on the model’s pretraining.

## Why retrieval is needed
LLMs don’t automatically know your internal data. Retrieval provides:
- **Grounding**: answers tied to source passages
- **Freshness**: updated content without retraining
- **Traceability**: citations to retrieved chunks
- **Control**: you decide what documents are searchable

## Core pipeline
**Documents → Chunk → Embed → Index → Query Embed → Nearest Neighbors → Retrieved passages**

In this notebook, we build this pipeline end-to-end using Azure OpenAI embeddings + a local FAISS vector store.


# 2) Load your data (hands-on)

Run the lab in two modes:
- **Option A (default)**: built-in mini corpus
- **Option B**: load `.txt` files from `./data`

> For real “own data”, use Option B.


In [4]:
# Option A: built-in mini corpus

documents = [
    {
        'doc_id': 'doc1',
        'title': 'Tokenization basics',
        'text': (
            'Tokenization converts text into tokens that models process. '
            'Token counts affect cost, latency, and context limits. '
            'Token-aware chunking helps keep inputs within limits.'
        )
    },
    {
        'doc_id': 'doc2',
        'title': 'Embeddings overview',
        'text': (
            'Embeddings map text to vectors so meaning can be compared. '
            'Cosine similarity is commonly used to compare embedding vectors. '
            'Embeddings are used for semantic search, clustering, and retrieval.'
        )
    },
    {
        'doc_id': 'doc3',
        'title': 'Vector stores',
        'text': (
            'Vector stores index embeddings to enable fast nearest-neighbor search. '
            'They store embeddings plus metadata and support filtering and retrieval. '
            'FAISS is a popular library for local vector indexing.'
        )
    },
]

print('Loaded docs (Option A):', len(documents))

Loaded docs (Option A): 3


**OPTIONAL. PLEASE SKIP THIS CELL IF YOU ARE RUNNING THE LAB ON BUILT-IN CORPUS MODE**

In [5]:
# Option B: load your own .txt files from ./data (overwrites Option A if folder exists)

from pathlib import Path

DATA_DIR = Path('data')  # create this folder and add .txt files

if DATA_DIR.exists():
    docs = []
    for p in DATA_DIR.glob('*.txt'):
        docs.append({
            'doc_id': p.name,
            'title': p.stem,
            'text': p.read_text(encoding='utf-8', errors='ignore')
        })
    if docs:
        documents = docs

print('Docs after Option B (if folder exists):', len(documents))
if documents:
    print('Sample doc:', documents[0]['doc_id'], '-', documents[0]['title'])

Docs after Option B (if folder exists): 3
Sample doc: 01_ai_usage_guidelines.txt - 01_ai_usage_guidelines


# 3) Token-aware chunking

We chunk by **tokens** to avoid exceeding model limits.

Typical starting defaults:
- Chunk size: 200–800 tokens
- Overlap: 10–20% of chunk size

We’ll use `tiktoken` for tokenization.


In [6]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')
print('Token encoding:', enc.name)

Token encoding: cl100k_base


In [7]:
from typing import List

def chunk_by_tokens(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    assert chunk_size > 0
    assert 0 <= overlap < chunk_size

    ids = enc.encode(text)
    chunks = []
    start = 0
    while start < len(ids):
        end = min(start + chunk_size, len(ids))
        chunks.append(enc.decode(ids[start:end]))
        if end == len(ids):
            break
        start = end - overlap
    return chunks

# Build chunks + metadata
chunked = []
for d in documents:
    parts = chunk_by_tokens(d['text'], chunk_size=400, overlap=50)
    for i, c in enumerate(parts):
        chunked.append({
            'doc_id': d['doc_id'],
            'title': d['title'],
            'chunk_id': f"{d['doc_id']}::chunk{i:04d}",
            'text': c,
            'tokens': len(enc.encode(c)),
        })

print('Chunks created:', len(chunked))
chunked[:2]

Chunks created: 3


[{'doc_id': '01_ai_usage_guidelines.txt',
  'title': '01_ai_usage_guidelines',
  'chunk_id': '01_ai_usage_guidelines.txt::chunk0000',
  'text': 'AI Usage Guidelines (Internal Knowledge Base Excerpt)\nVersion: 1.3 | Last updated: 2025-09-18\n\nPurpose\nThese guidelines describe how teams may use AI tools for drafting, summarization, coding assistance, and analytics,\nwhile maintaining security, privacy, and quality standards.\n\nKey principles\n1) Data classification\n- Public: may be used freely.\n- Internal: may be used only with approved AI services and access controls.\n- Confidential/Restricted: must not be sent to external services unless explicitly approved.\n\n2) Prompt hygiene\n- Do not include secrets, API keys, passwords, or private customer identifiers in prompts.\n- Use placeholders (e.g., <API_KEY>, <CUSTOMER_ID>) and secure configuration stores.\n\n3) Human review\n- AI outputs must be reviewed for accuracy, bias, and completeness before use in production materials.\n- Cl

# 4) Build your vector store (FAISS)

We will:
1) Embed each chunk using Azure OpenAI
2) Normalize vectors (so cosine similarity becomes dot product)
3) Build a FAISS index (`IndexFlatIP`)

> `IndexFlatIP` is exact search (perfect for labs). For large corpora, use approximate indexes.


In [8]:
import numpy as np

def embed_texts(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=EMBEDDINGS_DEPLOYMENT, input=batch)
        vectors.extend([r.embedding for r in resp.data])
    return np.array(vectors, dtype=np.float32)

texts = [c['text'] for c in chunked]
X = embed_texts(texts)
print('Embeddings matrix:', X.shape)

Embeddings matrix: (3, 1536)


In [9]:
import sys
print(sys.executable)

c:\Python\python.exe


In [10]:
!python -m pip install -U pip
!python -m pip install faiss-cpu

In [11]:
import faiss

# Normalize for cosine similarity
faiss.normalize_L2(X)

dim = X.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors == cosine similarity
index.add(X)

print('FAISS index built')
print('Vectors indexed:', index.ntotal)

FAISS index built
Vectors indexed: 3


# 5) Query the vector store (semantic search)

We will:
- embed the query
- search nearest neighbors
- display top-k results with scores

Interpretation:
- Scores are cosine similarities (higher = more similar).


In [12]:
def search(query: str, top_k: int = 5):
    q = embed_texts([query])
    faiss.normalize_L2(q)
    scores, ids = index.search(q, min(top_k, index.ntotal))

    results = []
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        item = chunked[int(idx)]
        results.append({
            'rank': rank,
            'score': float(score),
            'chunk_id': item['chunk_id'],
            'title': item['title'],
            'doc_id': item['doc_id'],
            'tokens': item['tokens'],
            'text_preview': item['text'][:350] + ('...' if len(item['text']) > 350 else ''),
        })
    return results

results = search('How do embeddings help semantic search?', top_k=5)
results

[{'rank': 1,
  'score': 0.7980833053588867,
  'chunk_id': '03_vector_store_best_practices.txt::chunk0000',
  'title': '03_vector_store_best_practices',
  'doc_id': '03_vector_store_best_practices.txt',
  'tokens': 286,
  'text_preview': 'Vector Store Best Practices (Notes)\nLast updated: 2025-08-11\n\nWhy a vector store\nA vector store enables fast similarity search over embeddings. This is the core building block behind semantic search\nand Retrieval-Augmented Generation (RAG).\n\nIndexing recommendations\n- Normalize vectors if you want cosine similarity (cosine = dot product on normaliz...'},
 {'rank': 2,
  'score': 0.7755142450332642,
  'chunk_id': '02_rag_pipeline_runbook.txt::chunk0000',
  'title': '02_rag_pipeline_runbook',
  'doc_id': '02_rag_pipeline_runbook.txt',
  'tokens': 357,
  'text_preview': 'RAG Pipeline Runbook (Simplified)\nVersion: 2.0 | Last updated: 2025-10-02\n\nObjective\nEnable users to query internal documentation and receive grounded answers with citations.\n

In [13]:
for r in results:
    print('-'*90)
    print(f"#{r['rank']} | score={r['score']:.4f} | {r['title']} | {r['chunk_id']} | tokens={r['tokens']}")
    print(r['text_preview'])

------------------------------------------------------------------------------------------
#1 | score=0.7981 | 03_vector_store_best_practices | 03_vector_store_best_practices.txt::chunk0000 | tokens=286
Vector Store Best Practices (Notes)
Last updated: 2025-08-11

Why a vector store
A vector store enables fast similarity search over embeddings. This is the core building block behind semantic search
and Retrieval-Augmented Generation (RAG).

Indexing recommendations
- Normalize vectors if you want cosine similarity (cosine = dot product on normaliz...
------------------------------------------------------------------------------------------
#2 | score=0.7755 | 02_rag_pipeline_runbook | 02_rag_pipeline_runbook.txt::chunk0000 | tokens=357
RAG Pipeline Runbook (Simplified)
Version: 2.0 | Last updated: 2025-10-02

Objective
Enable users to query internal documentation and receive grounded answers with citations.

High-level flow
1) Ingest documents
- Sources: knowledge base articles, SOPs, 

# 6) Metadata filtering (query within a subset)

In real systems, you often restrict retrieval by:
- document type, title, tags
- department/product
- time/version
- access control (ACL)

We’ll implement **pre-filtering** by building a temporary sub-index.


In [14]:
import pandas as pd
chunk_df = pd.DataFrame(chunked)
chunk_df[['doc_id','title','chunk_id','tokens']].head()

,doc_id,title,chunk_id,tokens
0,01_ai_usage_guidelines.txt,01_ai_usage_guidelines,01_ai_usage_guidelines.txt::chunk0000,315
1,02_rag_pipeline_runbook.txt,02_rag_pipeline_runbook,02_rag_pipeline_runbook.txt::chunk0000,357
2,03_vector_store_best_practices.txt,03_vector_store_best_practices,03_vector_store_best_practices.txt::chunk0000,286


In [15]:
import numpy as np
from typing import Optional

def build_subindex(filtered_indices: np.ndarray):
    X_sub = X[filtered_indices].copy()
    faiss.normalize_L2(X_sub)
    sub = faiss.IndexFlatIP(X_sub.shape[1])
    sub.add(X_sub)
    return sub

def search_with_filter(query: str, top_k: int = 5, title_contains: Optional[str] = None, doc_id_in: Optional[list] = None):
    mask = np.ones(len(chunked), dtype=bool)
    if title_contains:
        mask &= chunk_df['title'].str.contains(title_contains, case=False, na=False).values
    if doc_id_in:
        mask &= chunk_df['doc_id'].isin(doc_id_in).values

    filtered_idx = np.where(mask)[0]
    if len(filtered_idx) == 0:
        return []

    q = embed_texts([query])
    faiss.normalize_L2(q)

    sub = build_subindex(filtered_idx)
    scores, ids = sub.search(q, min(top_k, len(filtered_idx)))

    results = []
    for rank, (sub_i, score) in enumerate(zip(ids[0], scores[0]), start=1):
        orig_i = int(filtered_idx[int(sub_i)])
        item = chunked[orig_i]
        results.append((orig_i, float(score), item))
    return results

filtered_results = search_with_filter('How do we compare vectors?', top_k=5, title_contains='embed')
for _, score, item in filtered_results:
    print('-'*90)
    print(f"score={score:.4f} | {item['chunk_id']} | {item['title']} | tokens={item['tokens']}")
    print(item['text'][:250] + ('...' if len(item['text']) > 250 else ''))

# 7) Retrieval quality improvements

We’ll add practical improvements:
1) Light normalization
2) Remove too-short chunks
3) Remove duplicates
4) Rebuild a filtered index
5) MMR diversification (reduce redundancy in top-k)


In [16]:
import re, hashlib

def normalize_text(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.sha256(s.encode('utf-8', errors='ignore')).hexdigest()

MIN_TOKENS = 20

seen = set()
filtered_chunked = []

for c in chunked:
    t = normalize_text(c['text'])
    tok = len(enc.encode(t))
    if tok < MIN_TOKENS:
        continue
    h = stable_hash(t)
    if h in seen:
        continue
    seen.add(h)

    cc = dict(c)
    cc['text'] = t
    cc['tokens'] = tok
    filtered_chunked.append(cc)

print('Original chunks:', len(chunked))
print('After filtering:', len(filtered_chunked))

Original chunks: 3
After filtering: 3


In [17]:
texts_f = [c["text"] for c in filtered_chunked]
if not texts_f:
    raise ValueError("No chunks left after filtering")

X_f = np.ascontiguousarray(np.array(embed_texts(texts_f), dtype="float32"))
faiss.normalize_L2(X_f)  # in‑place

index_f = faiss.IndexFlatIP(X_f.shape[1])
index_f.add(X_f)
print("Filtered index size:", index_f.ntotal)


Filtered index size: 3


In [18]:
def mmr_select(query_vec, candidate_vecs, candidate_items, k=5, lambda_=0.75):
    """MMR balances relevance and diversity."""
    selected = []
    selected_vecs = []

    rel = candidate_vecs @ query_vec  # cosine via normalized vectors

    for _ in range(min(k, len(candidate_items))):
        if not selected:
            best = int(np.argmax(rel))
            selected.append(best)
            selected_vecs.append(candidate_vecs[best])
            continue

        sel_mat = np.vstack(selected_vecs)
        div = (candidate_vecs @ sel_mat.T).max(axis=1)

        mmr = lambda_ * rel - (1 - lambda_) * div
        mmr[selected] = -1e9  # block already-selected

        best = int(np.argmax(mmr))
        selected.append(best)
        selected_vecs.append(candidate_vecs[best])

    return [candidate_items[i] for i in selected]

def search_mmr(query: str, fetch_k: int = 20, final_k: int = 5, lambda_: float = 0.75):
    q = embed_texts([query])
    faiss.normalize_L2(q)

    scores, ids = index_f.search(q, min(fetch_k, index_f.ntotal))
    cand_ids = ids[0]
    cand_scores = scores[0]

    cand_vecs = X_f[cand_ids]
    cand_items = []
    for idx, score in zip(cand_ids, cand_scores):
        item = filtered_chunked[int(idx)]
        cand_items.append((int(idx), float(score), item))

    selected = mmr_select(q[0], cand_vecs, cand_items, k=final_k, lambda_=lambda_)
    return selected

mmr_results = search_mmr('Explain token-aware chunking and why it matters.', fetch_k=25, final_k=5, lambda_=0.75)
for _, score, item in mmr_results:
    print('-'*90)
    print(f"score={score:.4f} | {item['chunk_id']} | {item['title']} | tokens={item['tokens']}")
    print(item['text'][:260] + ('...' if len(item['text']) > 260 else ''))

------------------------------------------------------------------------------------------
score=0.7995 | 03_vector_store_best_practices.txt::chunk0000 | 03_vector_store_best_practices | tokens=278
Vector Store Best Practices (Notes) Last updated: 2025-08-11 Why a vector store A vector store enables fast similarity search over embeddings. This is the core building block behind semantic search and Retrieval-Augmented Generation (RAG). Indexing recommendat...
------------------------------------------------------------------------------------------
score=0.7725 | 02_rag_pipeline_runbook.txt::chunk0000 | 02_rag_pipeline_runbook | tokens=339
RAG Pipeline Runbook (Simplified) Version: 2.0 | Last updated: 2025-10-02 Objective Enable users to query internal documentation and receive grounded answers with citations. High-level flow 1) Ingest documents - Sources: knowledge base articles, SOPs, runbooks...
------------------------------------------------------------------------------------------

# 8) Grounding-ready retrieval bundle (citations)

Many RAG systems need retrieval results packaged with:
- stable citation ids (e.g., `[S1]`, `[S2]`)
- document metadata
- the chunk text
- scores

We’ll build a function that returns a **grounding bundle**.


In [19]:
from typing import Dict, Any

def grounding_bundle(query: str, top_k: int = 5) -> Dict[str, Any]:
    q = embed_texts([query])
    faiss.normalize_L2(q)

    scores, ids = index_f.search(q, min(top_k, index_f.ntotal))

    hits = []
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        item = filtered_chunked[int(idx)]
        hits.append({
            'rank': rank,
            'score': float(score),
            'citation_id': f"[S{rank}]",
            'doc_id': item.get('doc_id'),
            'title': item.get('title'),
            'chunk_id': item.get('chunk_id'),
            'text': item.get('text'),
            'tokens': item.get('tokens'),
        })

    return {'query': query, 'results': hits}

bundle = grounding_bundle('How does cosine similarity work for embeddings?', top_k=5)
bundle['results'][0].keys()

dict_keys(['rank', 'score', 'citation_id', 'doc_id', 'title', 'chunk_id', 'text', 'tokens'])

In [20]:
def print_bundle(bundle: Dict[str, Any], preview_chars: int = 450):
    print('QUERY:', bundle['query'])
    for r in bundle['results']:
        print('\n' + '-'*90)
        print(f"{r['citation_id']} score={r['score']:.4f} | {r['title']} | {r['chunk_id']} | tokens={r['tokens']}")
        txt = r['text'] or ''
        print(txt[:preview_chars] + ('...' if len(txt) > preview_chars else ''))

print_bundle(bundle)

QUERY: How does cosine similarity work for embeddings?

------------------------------------------------------------------------------------------
[S1] score=0.8251 | 03_vector_store_best_practices | 03_vector_store_best_practices.txt::chunk0000 | tokens=278
Vector Store Best Practices (Notes) Last updated: 2025-08-11 Why a vector store A vector store enables fast similarity search over embeddings. This is the core building block behind semantic search and Retrieval-Augmented Generation (RAG). Indexing recommendations - Normalize vectors if you want cosine similarity (cosine = dot product on normalized vectors). - Keep a stable mapping between the vector row and its metadata (doc_id, chunk_id, title,...

------------------------------------------------------------------------------------------
[S2] score=0.7736 | 02_rag_pipeline_runbook | 02_rag_pipeline_runbook.txt::chunk0000 | tokens=339
RAG Pipeline Runbook (Simplified) Version: 2.0 | Last updated: 2025-10-02 Objective Enable users 

# 9) Optional: grounded answer generation (gpt-4o-mini)

This step uses retrieval results as **sources** and asks the model to answer **only from those sources**, returning citations like `[S1]` inline.

> If your environment doesn’t have a chat deployment configured, you can skip this section.


In [21]:
def grounded_answer(query: str, top_k: int = 5) -> str:
    b = grounding_bundle(query, top_k=top_k)

    sources_text = '\n\n'.join(
        f"{r['citation_id']} {r['title']} | {r['chunk_id']}\n{r['text']}"
        for r in b['results']
    )

    messages = [
        {'role': 'system', 'content': 'You are a precise instructor. Answer only from the provided sources. If unsure, say you do not know.'},
        {'role': 'user', 'content': (
            f"Question: {query}\n\nSources:\n{sources_text}\n\n"
            "Instructions:\n- Use only the sources above.\n- Add citations like [S1] inline.\n- Keep it concise."
        )},
    ]

    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=messages,
        temperature=0.2,
    )
    return resp.choices[0].message.content

# Example (uncomment to run):
# print(grounded_answer('What is token-aware chunking and why is it used?', top_k=5))

# 10) Persist (save/load) the vector store

In real projects, you typically don’t re-embed everything each run.
We will persist:
- FAISS index
- chunk metadata (JSONL)
- embeddings matrix (NumPy)

This makes the lab “resume-friendly”.


In [22]:
import json
from pathlib import Path

OUT_DIR = Path('vector_store_out')
OUT_DIR.mkdir(exist_ok=True)

faiss.write_index(index_f, str(OUT_DIR / 'faiss.index'))

with open(OUT_DIR / 'chunks.jsonl', 'w', encoding='utf-8') as f:
    for c in filtered_chunked:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')

np.save(OUT_DIR / 'embeddings.npy', X_f)

print('Saved vector store to:', OUT_DIR.resolve())

Saved vector store to: \\260302aaibatch.file.core.windows.net\20260302acclaixlabfiles\Day2\Lab1\vector_store_out


In [23]:
def load_vector_store(path='vector_store_out'):
    path = Path(path)
    idx = faiss.read_index(str(path / 'faiss.index'))
    X_loaded = np.load(path / 'embeddings.npy').astype(np.float32)

    chunks_loaded = []
    with open(path / 'chunks.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            chunks_loaded.append(json.loads(line))

    return idx, X_loaded, chunks_loaded

index_loaded, X_loaded, chunks_loaded = load_vector_store(OUT_DIR)
print('Loaded:', index_loaded.ntotal, 'vectors;', len(chunks_loaded), 'chunks')

Loaded: 3 vectors; 3 chunks


# 11) Quick sanity evaluation

A lightweight evaluation pattern for labs:
- Write a few questions
- Define expected keywords/topics
- Check whether top-k contains the expected signal

This isn’t a full benchmark, but it helps validate that retrieval is working.


In [24]:
eval_questions = [
    ('What is tokenization?', 'token'),
    ('How do embeddings help semantic search?', 'embed'),
    ('Why chunk by tokens?', 'chunk'),
    ('What is cosine similarity used for?', 'cosine'),
]

def simple_eval(questions, top_k=3):
    rows = []
    for q, expected_keyword in questions:
        b = grounding_bundle(q, top_k=top_k)
        hit = any(
            expected_keyword.lower() in (((r.get('text') or '').lower() + ' ' + (r.get('title') or '').lower()))
            for r in b['results']
        )
        rows.append({'question': q, 'expected_keyword': expected_keyword, 'top_k_hit': hit})
    return pd.DataFrame(rows)

simple_eval(eval_questions, top_k=3)

,question,expected_keyword,top_k_hit
0,What is tokenization?,token,True
1,How do embeddings help semantic search?,embed,True
2,Why chunk by tokens?,chunk,True
3,What is cosine similarity used for?,cosine,True


---

## Next enhancements (optional)
- Add PDF ingestion (extract text → chunk → embed)
- Add a stronger reranker (cross-encoder) after retrieval
- Swap FAISS for a managed vector store (Azure AI Search / Cosmos DB / etc.)
- Add ACL-aware retrieval (security trimming)

---

### End of Notebook
You now have an end-to-end pipeline for **querying your own data** and **building your vector store**, including filtering, quality improvements, and grounding-ready outputs.
